# Aufgabe 1a) - CNN zur Erkennung von Autos mit CIFAR-10

In [ ]:
# Bibliotheken importieren
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.datasets import cifar10

print(tf.__version__)

In [ ]:
# CIFAR-10 Datensatz laden
# Enthaelt 60.000 Bilder in 10 Klassen (32x32 Pixel, RGB)
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

print(f'Trainingsbilder: {X_train.shape}')
print(f'Testbilder: {X_test.shape}')

In [ ]:
# Autos haben in CIFAR-10 den Label 1
# y_train_car ist True wenn ein Auto auf dem Bild ist, sonst False
y_train_car = y_train == 1
y_test_car = y_test == 1

# Pixelwerte normalisieren: 0-255 -> 0.0-1.0, sonst wird Gradient zu grob bei hohen Werten
X_train = X_train.astype(np.float32) / 255
X_test = X_test.astype(np.float32) / 255

print(f'Autos im Training: {y_train_car.sum()} von {len(y_train_car)}')

In [ ]:
# CNN Modell aufbauen
model = Sequential()

# Erste Faltungsschicht mit 32 Filtern
# padding=same damit das Bild nicht kleiner wird (kein Informationsverlust)
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Zweite Faltungsschicht mit 64 Filtern
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Flatten + vollverbundene Schichten
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))  # Dropout gegen Overfitting

# Ausgabe: 1 Neuron mit Sigmoid fuer binaere Klassifikation (Auto ja/nein)
model.add(Dense(1, activation='sigmoid'))

model.summary()

In [ ]:
# Modell kompilieren
# binary_crossentropy weil wir nur 2 Klassen haben (Auto / kein Auto)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Training
history = model.fit(
    X_train, y_train_car,
    epochs=10,
    batch_size=32,
    validation_split=0.1  # 10% der Daten zum Validieren
)

In [ ]:
# Trainingsverlauf plotten
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validierung')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
# Modell auf Testdaten testen
loss, acc = model.evaluate(X_test, y_test_car)
print(f'Test Accuracy: {acc*100:.2f}%')

In [ ]:
# Modell abspeichern
model.save('cardetector.h5')
print('Modell gespeichert als cardetector.h5')